# Type Search
**Table of contents**

- Overview
- Setup
  - Authentication Token
- Query
  - Output Description
- Related Links

## Overview
This notebook demonstrates how to use the Type Search API to discover activity types using semantic search. Unlike traditional keyword matching, semantic search understands the meaning and context of your query. It allows users to:

- Search for activity types using natural language (e.g., "car" matches "passenger vehicles")
- Filter results by location (country and state/province)
- Filter by date to get factors valid for specific time periods
- Navigate through paginated results
- View confidence scores showing relevance of each match

The Type Search API is valuable for discovery and exploration, helping users find the right activity types before performing emissions calculations. Results are ranked by relevance, making it easy to identify the most appropriate activity types for your needs. This is particularly useful when you're unsure of the exact terminology used in emissions databases or when exploring what activity types are available for a specific region or time period.

## Setup

Ensure that you have Python installed in your system. Python 3+ is required.

<b>Note:</b> To run this notebook, you must first add your credentials to `config.read('../../../auth/secrets.ini')` in the following format:

```
[EAPI]
api.pat_token = <Your Emissions API PAT token>
api.client_id = <Your GHG APIs client Id>
```

In [ ]:
# Install the packages below using pip/pip3 based on your python version.
%pip install pandas configparser IPython requests

In [1]:
import pandas as pd
import configparser
import requests
import json
from IPython.display import display as display_summary

### Authentication Token

Run the following code snippet to generate the Auth Bearer Token by using your api_key configured in secrets.ini.

In [2]:
config = configparser.RawConfigParser()
config.read(['../../../auth/secrets.ini','../../../auth/config.ini'])

EAPI_PAT_TOKEN      = config.get('EAPI', 'api.pat_token')
EAPI_TENANT_ID      = config.get('EAPI', 'api.tenant_id')

EAPI_AUTH_ENDPOINT  = config.get('EAPI', 'api.auth_endpoint')
EAPI_BASE_URL       = config.get('EAPI', 'api.base_url')
EAPI_ENDPOINT       = f"{EAPI_BASE_URL}/recommender/type/search"

EAPI_AUTH_CLIENT_ID = 'saascore-' + EAPI_TENANT_ID
EAPI_CLIENT_ID      = 'ghgemissions-' + EAPI_TENANT_ID

auth_request_headers: dict = {}
auth_request_headers["X-IBM-Client-Id"] = EAPI_AUTH_CLIENT_ID
auth_request_headers["X-IBM-Envizi-Pat"] = EAPI_PAT_TOKEN

verify = True

auth_url = f"{EAPI_AUTH_ENDPOINT}"
              
response = requests.post(url = auth_url,
                        headers = auth_request_headers,
                        verify  = verify
                       )

if response.status_code == 200:
    jwt_token = response.text
    print("Authentication Success")
else:     
    print("Authentication Failed")
    print(response.text)

Authentication Success


## Query
The example request payload demonstrates how to search for activity types using IBM Envizi - Emissions API by specifying location (USA), a search term ("electricity"), pagination parameters (page 1, 20 results), and a reference date (January 4, 2025).

The optional `enableReranker` flag (boolean, default `true`) controls whether a cross-encoder reranking step refines the result ordering. Set it to `false` to rank purely by semantic similarity.

In [11]:
payload = {
  "location": {
    "country": "USA"
  },
  "activity": {
    "search": "electricity"
  },
  "pagination": {
    "page": 1,
    "size": 20
  },
  "time": {
    "date": "2025-01-04"
  },
  "enableReranker": True
}

In [12]:
# Create the query headers
request_headers: dict = {}
request_headers["Content-Type"] = "application/json"
request_headers["x-ibm-client-id"] = EAPI_CLIENT_ID
request_headers["Authorization"] = "Bearer " + jwt_token
request_headers["X-Org-Id"] = EAPI_ORG_ID

# Submit the request
response = requests.post(EAPI_ENDPOINT, 
                         headers = request_headers, 
                         data = json.dumps(payload))

For more information about allowable parameters for the payload, please see [Emissions API Developer Guide]().

In [13]:
if response.status_code == 200 and response.text:
    # Parse the JSON response
    response_json = response.json()
    
    # Extract the types list
    types_list = response_json.get('types', [])
    
    # Create dataframe from types list
    df_types = pd.json_normalize(types_list)
    
    # Display results
    print(f"\nFound {len(types_list)} matching activity types\n")
    
    pd.set_option('display.max_colwidth', None)
    display_summary(df_types)
else:
    print(f"Error: {response.status_code}")
    print(f"Response: {response.text}")


Found 20 matching activity types



,activityType,activityUnit,activityDescription,confidence
0,Electricity,"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm, usd]",Electricity consumption.,1.0000
1,Electricity - Green,"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",Use of green electricity.,0.6831
2,Electricity - Scope 3,"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",Use of Scope 3 indirect electricity.,0.6653
3,Copper Wire,"[g, kg, lb, million lb, st, t, thousand lb]",Use of copper wire.,0.6092
4,Waste - Electrical,"[g, kg, lb, million lb, st, t, thousand lb]",Generate electrical waste.,0.6066
5,Light bulbs,[usd],Expenditure on light bulbs.,0.6040
6,Utilities,[usd],Spend on utilities.,0.5872
7,Material Use - Electrical items - IT,"[g, kg, lb, million lb, st, t, thousand lb]",Use of information technology electrical equipment such as computers and related devices.,0.5660
8,Material Use - Electrical items - small,"[g, kg, lb, million lb, st, t, thousand lb]",Use of small electrical items such as household and personal devices.,0.5539
9,Coal (electricity generation),"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, g, kBTU, kWh, kg, lb, mWh, million lb, st, t, thm, thousand lb]",Coal used for generating electricity,0.5356


### Output Description

The Type Search API returns a list of activity types with the following fields:

<b>activityType</b> - The name/description of the activity type (e.g., "Personal Vehicles - EV", "Electricity for EVs - Executive - Battery Electric")

<b>activityUnit</b> - List of available units for this activity type (e.g., ["ft", "km", "m", "mi", "nmi"] for distance-based activities, ["kWh", "MWh"] for electricity)

<b>activityDescription</b> - Detailed description of the activity type explaining what it represents (e.g., "Use of electric personal vehicle.", "Use of electricity to charge an executive-class battery electric car.")

<b>confidence</b> - Relevance score (0.0 to 1.0) indicating how well the activity type matches your search query. Higher values indicate better matches:
- 0.9 - 1.0: Excellent match (exact or very close semantic match)
- 0.7 - 0.9: Good match (related concepts)
- 0.5 - 0.7: Fair match (somewhat related)

<b>scope</b> - List of distinct GHG Protocol scope(s) associated with the activity type (e.g., ["2"] for Scope 2, ["1", "2"] when multiple scopes apply, "3.1"-"3.15" for Scope 3 categories). Omitted from the response when no scope information is available.

Results are automatically sorted by confidence score in descending order, with the most relevant matches appearing first.

## Related Links
[Emissions API Developer Guide](https://developer.ibm.com/apis/catalog/ghgemissions--ibm-envizi-emissions-api/Introduction)